In [12]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [4]:
output = df["diagnosis"]
output.head()

0    M
1    M
2    M
3    M
4    M
Name: diagnosis, dtype: str

In [5]:
input = df.drop(columns=['diagnosis'])
input.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
X_train, X_test, y_train, y_test = train_test_split(input, output, test_size=0.2)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [15]:
#The error is caused by StandardScaler producing float64 features, 
# while PyTorch’s nn.Linear uses float32 weights by default.

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)


X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [10]:
X_train_tensor.shape

torch.Size([455, 30])

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):
    # u can transformation in here
    return self.features[index], self.labels[index]

In [20]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [21]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [22]:
import torch.nn as nn

class MySimpleNN(nn.Module):

  def __init__(self, nn_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(nn_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    y_pred = self.network(features)
    return y_pred


In [23]:
learning_rate = 0.1
epochs = 50

In [24]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()        

# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features)

    # loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.675090491771698
Epoch: 2, Loss: 0.31831711530685425
Epoch: 3, Loss: 0.34697070717811584
Epoch: 4, Loss: 0.39072588086128235
Epoch: 5, Loss: 0.3652912676334381
Epoch: 6, Loss: 0.25811225175857544
Epoch: 7, Loss: 0.3565217852592468
Epoch: 8, Loss: 0.16205336153507233
Epoch: 9, Loss: 0.12895742058753967
Epoch: 10, Loss: 0.13438251614570618
Epoch: 11, Loss: 0.26158806681632996
Epoch: 12, Loss: 0.13185082376003265
Epoch: 13, Loss: 0.1352708339691162
Epoch: 14, Loss: 0.07382742315530777
Epoch: 15, Loss: 0.10302041471004486
Epoch: 16, Loss: 0.06730060279369354
Epoch: 17, Loss: 0.06734383851289749
Epoch: 18, Loss: 0.015098131261765957
Epoch: 19, Loss: 0.08909199386835098
Epoch: 20, Loss: 0.13720382750034332
Epoch: 21, Loss: 0.10347563028335571
Epoch: 22, Loss: 0.05708727240562439
Epoch: 23, Loss: 0.06704161316156387
Epoch: 24, Loss: 0.020917270332574844
Epoch: 25, Loss: 0.041255395859479904
Epoch: 26, Loss: 0.08301401138305664
Epoch: 27, Loss: 0.13502173125743866
Epoch: 28, L

In [25]:
#This prints each layer’s weights and biases
for name, param in model.named_parameters():
    print(name, param.detach())

network.0.weight tensor([[ 0.3185,  0.0953,  0.3123,  0.1592, -0.0829,  0.0333, -0.0685,  0.1292,
          0.0340, -0.1825,  0.1781, -0.1485,  0.0249,  0.1905,  0.4639, -0.1123,
          0.1679,  0.2016, -0.2867, -0.2112,  0.1349,  0.4227,  0.0109,  0.1647,
          0.0876, -0.0194,  0.2025,  0.2263,  0.0463, -0.1636],
        [ 0.1617,  0.0944,  0.0921,  0.2052,  0.2008, -0.1285,  0.1899,  0.0338,
          0.0143, -0.2103,  0.3050,  0.0485,  0.3969,  0.2211, -0.2739, -0.3478,
         -0.2163,  0.1439, -0.3424, -0.1846,  0.2480,  0.1949,  0.1794,  0.4158,
          0.1868,  0.0724,  0.3504,  0.1055,  0.0824,  0.1610],
        [ 0.3434,  0.2215,  0.1972,  0.1580, -0.0080,  0.1803,  0.2098,  0.3711,
          0.0600, -0.1724,  0.3285, -0.0987, -0.0603,  0.3028, -0.1375, -0.2469,
          0.0183, -0.1305, -0.0244, -0.5279,  0.6303,  0.4817,  0.4432,  0.5384,
          0.4447,  0.1330,  0.4228,  0.6471,  0.4562,  0.0278]])
network.0.bias tensor([0.4497, 0.4728, 0.9148])
network.2.wei

In [31]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.5804


In [27]:
#%pip install torchinfo

In [28]:
from torchinfo import summary

summary(model=model, input_size=X_train_tensor.shape)

Layer (type:depth-idx)                   Output Shape              Param #
MySimpleNN                               [455, 1]                  --
├─Sequential: 1-1                        [455, 1]                  --
│    └─Linear: 2-1                       [455, 3]                  93
│    └─ReLU: 2-2                         [455, 3]                  --
│    └─Linear: 2-3                       [455, 1]                  4
│    └─Sigmoid: 2-4                      [455, 1]                  --
Total params: 97
Trainable params: 97
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.04
Input size (MB): 0.05
Forward/backward pass size (MB): 0.01
Params size (MB): 0.00
Estimated Total Size (MB): 0.07